# クロスセクションデータと時系列データ

データは、その並び方によって大きく2つに分けられる。  
ある一時点で多数の対象を横に並べたデータをクロスセクションデータ（横断面データ）、1つの対象を時間にそって並べたデータを時系列データという。

第11回では、都道府県データ（クロスセクション・横断）とGDPの推移（時系列）を対比し、時系列の見方に少し触れた。

今回はこの2つのデータ型を改めて整理したうえで、時系列の見方を新しく学ぶ。とくに、月ごとにくり返す動き（季節変動）と、変化のとらえ方（前年同月比・指数）に重点を置く。

題材は訪日外国人数である。  
日本を訪れる外国人の数は、近年の日本経済で注目される指標であり、国・地域による違い（横断面）と、時間に伴う変化（時系列）の両面をもつ。日本政府観光局（JNTO）の統計を用いる。

## 1. ライブラリの読み込み

これまでの回と同じライブラリを用いる。回帰分析には `statsmodels` を使う。

In [ ]:
%pip install japanize-matplotlib-jlite -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import japanize_matplotlib_jlite

## 2. 使用するデータ

日本政府観光局（JNTO）「訪日外客統計」より、訪日外国人数（訪日外客数）を用いる。訪日外客数とは、外国人が日本を訪れた延べ人数のことである。

今回は、同じ訪日外客数を3つの形で用意する。

- 国・地域別（2024年）：韓国・中国など主要な国・地域ごとの、2024年1年間の訪日客数。ある一時点で多数の対象を並べた横断面データである。
- 年ごと（2013〜2025年）：日本全体の訪日客数を、1年に1つずつ並べた時系列データである。
- 月ごと（2023〜2024年）：日本全体の訪日客数を、1か月に1つずつ並べた時系列データである。季節による動きを見るために月ごとの値を使う。

人数はそのままだと桁が大きいので、表やグラフでは万人（1万人＝10000人）に直して用いる。なお、直近の月の値は推計値であり、あとから確定値に直ることがある。

In [ ]:
# 国・地域別の訪日外客数（2024年、上位10の国・地域）
# 出典：日本政府観光局（JNTO）「訪日外客統計」訪日外客数（2024年12月および年間推計値）
# URL：https://www.jnto.go.jp/news/press/20250115_monthly.html
# 単位：人（2024年の1年間の合計）

country = ["韓国", "中国", "台湾", "米国", "香港",
           "タイ", "豪州", "シンガポール", "カナダ", "マレーシア"]

visitors_2024 = [8817765, 6981342, 6044316, 2724594, 2683391,
                 1148848, 920196, 691226, 579445, 506883]

df_country = pd.DataFrame({
    "country": country,
    "visitors": visitors_2024,
})

# 万人に直す
df_country["visitors_man"] = df_country["visitors"] / 10000

df_country

In [ ]:
# 日本全体の訪日外客数（年ごと、2013〜2025年）
# 出典：日本政府観光局（JNTO）「訪日外客統計」
# URL：https://www.jnto.go.jp/statistics/data/visitors-statistics/
# 単位：人（各年の1年間の合計）

year = list(range(2013, 2026))

visitors_year = [
    10363904, 13413467, 19737409, 24039700, 28691073,
    31191856, 31882049, 4115828, 245862, 3832110,
    25066350, 36869900, 42683600
]

df_year = pd.DataFrame({
    "year": year,
    "visitors": visitors_year,
})

# 万人に直す
df_year["visitors_man"] = df_year["visitors"] / 10000

df_year

月ごとのデータは、2023年と2024年の24か月ぶんある。表にするには、各行が何年の何月かを表す列も必要になる。この2つの列は、リストのくり返しと連結でつくる。

- `[2023]*12` は、`2023` を12個ならべたリスト `[2023, 2023, …, 2023]`（長さ12）をつくる。同じように `[2024]*12` は `2024` を12個ならべる。この2つを `+` でつなぐと、2023が12個・2024が12個ならんだ長さ24のリストになる。これが各行の「年」である。
- `list(range(1, 13))` は 1から12までのリスト `[1, 2, …, 12]` である。これを `*2` で2回くり返すと、1〜12・1〜12とならんだ長さ24のリストになる。これが各行の「月」である。

このように、リストは `*` で同じ内容をくり返し、`+` でつなぐことができる。これを使うと、24か月ぶんの年と月を短く用意できる。

In [ ]:
# 日本全体の訪日外客数（月ごと、2023年1月〜2024年12月）
# 出典：日本政府観光局（JNTO）「訪日外客統計」訪日外客数（各月の推計値・確定値）
# URL：https://www.jnto.go.jp/statistics/data/visitors-statistics/
# 単位：人（各月の合計）

month_year  = [2023]*12 + [2024]*12
month_num   = list(range(1, 13)) * 2

visitors_month = [
    # 2023年 1月〜12月
    1497472, 1475455, 1817616, 1949236, 1899176, 2073441,
    2320694, 2157190, 2184442, 2516623, 2440890, 2734115,
    # 2024年 1月〜12月
    2688478, 2788224, 3081781, 3043003, 3040294, 3140642,
    3292602, 2933381, 2872487, 3312193, 3187000, 3489800
]

df_month = pd.DataFrame({
    "year": month_year,
    "month": month_num,
    "visitors": visitors_month,
})

# 万人に直す
df_month["visitors_man"] = df_month["visitors"] / 10000

# 折れ線用に、年と月を1つの時間軸にまとめる（例：2024年7月 → 2024 + 6/12）
df_month["time"] = df_month["year"] + (df_month["month"] - 1) / 12

df_month.head()

## 3. クロスセクションデータと時系列データ（整理）

データはその並び方で2つに分けられる。今回のデータで、両者の形を見比べる。

クロスセクションデータ（横断面データ）は、ある一時点で多数の対象を横に並べたデータである。国・地域別（2024年）の訪日客数がこれにあたる。1行が1つの国・地域を表す。

時系列データは、1つの対象を時間にそって繰り返し観測して並べたデータである。年ごと（または月ごと）の訪日客数がこれにあたる。1行が1つの時点（年、または月）を表す。

In [ ]:
# クロスセクションデータ（国・地域別、2024年）：1行が1つの国・地域
df_country[["country", "visitors_man"]].head()

In [ ]:
# 時系列データ（年ごと、日本全体）：1行が1つの年
df_year[["year", "visitors_man"]].head()

2つの表は、1行が表すものが異なる。横断面データでは1行が1つの対象（国・地域）、時系列データでは1行が1つの時点（年）である。横断面では横に多数の対象がならび、関心は対象どうしの違いにある。時系列では横に多数の時点がならび、関心は時間に伴う変化にある。

横断面データでは対象の並び順に意味がなく、国を五十音順にしても金額順にしても分析は変わらない。一方、時系列データでは年や月に順序があり、入れ替えることはできない。前年と比べる、傾向を読み取る、といった見方は、順序のある時系列データだからこそ成り立つ。グラフにするときは、横断面では横軸に対象を、時系列では横軸に時間（年・月）をとる。

## 4. クロスセクションデータをよむ（国・地域別）

2024年に日本を訪れた外国人を国・地域別に棒グラフで比べる。ある一時点（2024年）で、対象（国・地域）どうしの違いを見るのが横断面データの読み方である。

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(df_country["country"], df_country["visitors_man"])
plt.title("国・地域別の訪日外客数（2024年、上位10）")
plt.xlabel("国・地域")
plt.ylabel("訪日客数（万人）")
plt.grid(True, axis="y")
plt.show()

2024年は、韓国・中国・台湾の順で多く、この3つで大きな割合を占める。国・地域によって訪日客数は大きく異なる。

どれくらいの割合を占めるかを見るために、構成比（シェア）を計算する。ここでは2024年の訪日客数の合計に対する、各国・地域の割合（％）を求める。

In [ ]:
# 2024年の訪日客数の合計（人）
total_2024 = 36869900

# 各国・地域の構成比（％）
df_country["share"] = df_country["visitors"] / total_2024 * 100

df_country[["country", "visitors_man", "share"]].round(1)

上位10の国・地域で、2024年の訪日客数全体のおよそ8割を占める。韓国だけで全体の約24％である。このように、横断面データでは対象ごとの大きさや、全体に占める割合を比べることに関心がある。

## 5. 時系列データをよむ①：年ごとの長期の動き

次に時系列データを見る。日本全体の訪日客数を、年ごとに折れ線で描く。横軸に年、縦軸に訪日客数をとる。

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(df_year["year"], df_year["visitors_man"], marker="o")
plt.title("訪日外客数の推移（年ごと）")
plt.xlabel("年")
plt.ylabel("訪日客数（万人）")
plt.grid(True)
plt.show()

## 6. 前年比を計算する

前の年とくらべて何％増えた（減った）かを前年比（前年比成長率）という。pandas では `pct_change()` で計算できる。（復習）

$$
\text{前年比}\;(\%) = \frac{\text{今年の値} - \text{前年の値}}{\text{前年の値}} \times 100
$$

In [ ]:
# 訪日客数の前年比（％）
df_year["growth"] = df_year["visitors"].pct_change() * 100

df_year[["year", "visitors_man", "growth"]].round(1)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(df_year["year"], df_year["growth"])
plt.title("訪日外客数の前年比")
plt.xlabel("年")
plt.ylabel("前年比（％）")
plt.axhline(0, color="black", linewidth=0.8)
plt.grid(True, axis="y")
plt.show()

2020年は前年比で約−87％、2021年はさらに約−94％と、2年続けて大きく減った。逆に2022年以降は前年比が大きなプラスになり、落ち込みからの回復の勢いを表している。2024年は前年比で約+47％、2025年は約+16％である。

前年比は、水準（実際の人数）ではなく変化の勢いに注目する見方である。回復が進むにつれて前年比のプラス幅は小さくなっており、勢いが落ち着いてきたことがわかる。前年の値が計算の基準になるため、最初の年（2013年）の前年比は計算できず、欠損（NaN）となる。

## 7. 時系列データをよむ②：月ごとの動きと季節変動

年ごとのデータでは見えないが、月ごとのデータにすると新しい動きが見えてくる。2023年1月から2024年12月までの訪日客数を、月ごとに折れ線で描く。横軸は時間（年）である。

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_month["time"], df_month["visitors_man"], marker="o", markersize=4)
plt.title("訪日外客数の推移（月ごと、2023〜2024年）")
plt.xlabel("年")
plt.ylabel("訪日客数（万人）")
plt.grid(True)
plt.show()

折れ線は、全体として右上がりに伸びながら、その中で細かく上下をくり返している。この動きは2つに分けて考えられる。

- 全体が右上がりなのは、コロナからの回復、および日本のインバウンド需要拡大というトレンド（長期の傾向）である。
- 1年の中で、同じような山と谷がくり返し現れる。このような、毎年決まった時期に現れる動きを季節変動という。

季節変動があるのは、旅行に向いた時期とそうでない時期があるからである。桜の春や紅葉の秋は訪日客が多く、真冬の1〜2月は少なくなりやすい。季節変動を見やすくするために、2023年と2024年を、同じ「月」の軸で重ねて描く。

In [ ]:
# 2023年と2024年を、横軸を「月」にして重ねて描く
m2023 = df_month[df_month["year"] == 2023]
m2024 = df_month[df_month["year"] == 2024]

plt.figure(figsize=(9, 5))
plt.plot(m2023["month"], m2023["visitors_man"], marker="o", label="2023年")
plt.plot(m2024["month"], m2024["visitors_man"], marker="o", label="2024年")
plt.title("訪日外客数の月ごとの動き（2023年・2024年）")
plt.xlabel("月")
plt.ylabel("訪日客数（万人）")
plt.xticks(range(1, 13))
plt.legend()
plt.grid(True)
plt.show()

2つの年の線は、水準は違うが形がよく似ている。どちらの年も、春（3〜4月）や夏から秋（7月、10月）に多く、1〜2月に少ない。この「毎年くり返す形」が季節変動である。

一方、2024年の線が全体として2023年より上にあるのは、回復というトレンドを表す。このように、月ごとの時系列には、トレンド（回復）と季節変動（毎年の山と谷）が同時に混ざっている。時系列を読むときは、この2つを分けて考えると見通しがよくなる。

## 8. 前年同月比：季節をそろえて比べる

月ごとのデータで「増えたか減ったか」を見るとき、前の月とくらべる（前月比）と、季節変動の影響が混ざってしまう。たとえば1月から2月にかけて減っても、それが不調なのか、単に毎年2月が少ないだけなのか区別できない。

そこで、1年前の同じ月とくらべる。これを前年同月比という。同じ月どうしを比べるので季節の条件がそろい、季節変動を除いて動きを見ることができる。

$$
\text{前年同月比}\;(\%) = \frac{\text{今年のその月の値} - \text{前年の同じ月の値}}{\text{前年の同じ月の値}} \times 100
$$

In [ ]:
# 12か月前（前年の同じ月）とくらべる
df_month["yoy"] = df_month["visitors"].pct_change(12) * 100

# 2024年の各月の前年同月比（2023年の同じ月とくらべた値）
yoy_2024 = df_month[df_month["year"] == 2024]

yoy_2024[["month", "visitors_man", "yoy"]].round(1)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(yoy_2024["month"], yoy_2024["yoy"])
plt.title("訪日外客数の前年同月比（2024年、2023年の同月とくらべて）")
plt.xlabel("月")
plt.ylabel("前年同月比（％）")
plt.xticks(range(1, 13))
plt.grid(True, axis="y")
plt.show()

2024年はすべての月で前年同月比がプラスであり、どの月も前年より訪日客が増えたことがわかる。とくに年の前半のプラス幅が大きい。これは、比べる相手である2023年の前半がまだ回復の途中で少なかったためである。年の後半になるほどプラス幅が小さくなるのは、2023年後半にはすでに回復が進んでいたからである。

前年同月比は、季節変動を気にせずに増減を比べられるため、報道などでもよく使われる。「今月の訪日客数は前年同月比〇％増」という言い方は、この前年同月比のことである。前年の同じ月が基準になるので、最初の12か月（2023年）の前年同月比は計算できず欠損となる。

## 9. 移動平均でならす（12か月移動平均）

季節変動を消して、水準の動き（トレンド）だけを見たいときは、移動平均を使う。移動平均とは、連続するいくつかの値をずらしながら平均していったものである。月ごとのデータでは、1年ぶんの12か月について平均をとる12か月移動平均を使う。12か月を平均すると、1年の中の山と谷が打ち消し合い、季節変動が消える。

In [ ]:
# 12か月移動平均
df_month["ma12"] = df_month["visitors_man"].rolling(12).mean()

plt.figure(figsize=(10, 5))
plt.plot(df_month["time"], df_month["visitors_man"], alpha=0.4, marker="o", markersize=3, label="月ごとの値")
plt.plot(df_month["time"], df_month["ma12"], marker="o", markersize=4, label="12か月移動平均")
plt.title("訪日外客数と12か月移動平均")
plt.xlabel("年")
plt.ylabel("訪日客数（万人）")
plt.legend()
plt.grid(True)
plt.show()

細い線（月ごとの値）は季節変動で上下しているが、太い線（12か月移動平均）はなめらかに右上がりになっている。季節の山と谷が消え、回復というトレンドだけが残る。12か月ぶんの平均をとるため、最初の11か月は値が計算できず欠損となり、移動平均の線は途中から始まる。

このように、同じ時系列でも、前年同月比は「季節をそろえて増減を見る」道具、移動平均は「季節をならして水準の傾向を見る」道具として使い分けられる。

## 10. トレンドを直線でとらえる（回帰との接続）

これまではグラフや簡単な計算でトレンドを読み取ってきた。回帰分析を使うと、トレンドを直線として表すこともできる。時間を説明変数、訪日客数を被説明変数とした回帰をトレンド回帰という。

ここでは、動きが素直に右上がりだったコロナ前の2013〜2019年に絞って、直線をあてはめる。説明変数には、2013年からの経過年数（2013年を0、2014年を1、…）を用いる。

$$
（訪日客数） = \alpha + \beta \times （経過年数）
$$

傾き $\beta$ は「1年あたり訪日客数がどれだけ増えたか」を表す。

In [ ]:
# コロナ前の2013〜2019年だけを取り出す
pre = df_year[df_year["year"] <= 2019].copy()

# 2013年からの経過年数（2013→0, 2014→1, …）
pre["t"] = pre["year"] - 2013

# 被説明変数（訪日客数、万人）と説明変数（経過年数）
Y = pre["visitors_man"]
X = pre[["t"]]
X = sm.add_constant(X)   # 切片に対応する定数項を追加する

model = sm.OLS(Y, X)
result = model.fit()

result.params

In [ ]:
# 推定された直線を、実際のデータに重ねて描く
t_line = np.linspace(pre["t"].min(), pre["t"].max(), 100)
y_line = result.params["const"] + result.params["t"] * t_line
year_line = t_line + 2013

plt.figure(figsize=(9, 5))
plt.plot(pre["year"], pre["visitors_man"], "o", label="実際のデータ（2013〜2019年）")
plt.plot(year_line, y_line, color="tab:red", label="トレンド回帰の直線")
plt.title("コロナ前の訪日外客数とトレンド回帰")
plt.xlabel("年")
plt.ylabel("訪日客数（万人）")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 傾き（1年あたりの増加、万人）と決定係数
print("傾き（1年あたりの増加）:", round(result.params["t"], 1), "万人")
print("決定係数 R^2:", round(result.rsquared, 4))

傾きは約390万人であり、コロナ前は1年あたりおよそ390万人ずつ訪日客が増えていた、と読める。決定係数 $R^2$ は1に近く、この期間の動きは直線でよくあてはまる。

ただし、この直線が使えるのはコロナ前の期間に限られる。2020年以降は感染拡大という大きな出来事で動きが途切れ、同じ直線では説明できない。時系列は、こうした構造の変化やショックを含むことが多く、1本の直線だけで将来まで延ばして予測することには注意が必要である。トレンド回帰は、あくまである期間の傾向を要約する道具として使う。

## 11. まとめ

データは並び方によって、クロスセクションデータ（ある一時点で多数の対象を並べたもの）と時系列データ（1つの対象を時間にそって並べたもの）に分けられる。  横断面では対象どうしの違いに、時系列では時間に伴う変化に関心がある。時系列は順序をもち、行を入れ替えることはできない。

時系列には、長期の傾向（トレンド）、毎年くり返す季節変動、その時かぎりの不規則変動が混ざる。これらを読み解くために、前年比・前年同月比（季節をそろえて増減を見る）、移動平均（季節をならして傾向を見る）といった道具を使い分ける。さらに、時間を説明変数とした回帰（トレンド回帰）で、傾向を直線として表すこともできる。